In [1]:
# Transformer 모델 구축 - Transformer RAG(Retriever Augmentation Generation) 구성 및 모델 파이프라인 구축(데이터 수집기 + Qdrant 의미 기반 검색엔진 적용)
# - Retrieval 단계: Qdrant 의미 기반 검색엔진을 통해 관련 문서를 빠르게 찾아냄
# - Augmentation 단계: 검색된 문서를 기반으로 LLM 입력 프롬프트를 강화 → 맥락 있는 답변 생성
# - Generation 단계: LLM이 최종 답변을 생성 → 단순 생성보다 정확성과 신뢰성이 높아짐
# - 예시) 구글, 네이버에서 사용되는 RAG 시스템 구현

# 학습 목표 - 실무에서 사용되는 파이프라인 이해 및 적용
# 1. 데이터 수집기
# - 수집 대상 도메인: Google News RSS
# - RSS 피드 파싱: feedparser 라이브러리로 RSS XML을 읽고 기사 항목 추출
# - 데이터 정제: title,link,description,pubDate,soure 필드 추출, pubDate -> Python datetime 변화
# - DB 저장: PostgreSQL 테이블 news_articles 에 Insert
# - 수집 데이터 -> PostgreSQL 테이블 생성 및 입력
# - DB 조회 + 로깅 설정으로 데이터 관리

# 2. Qdrant 의미 기반 검색 구축
# - Qdrant 구축 + 뉴스 컬렉션 생성
# - Qdrant 검색엔진 서버 실행(qdrant.exe): .\LLM\qdrant\qdrant.exe
# - https://github.com/qdrant/qdrant/releases 공식사이트: qdrant-x86_64-pc-windows-msvc.zip, 압축해제 qdrant.exe 실행
# - API(curl) 테스트: http://localhost:6333/collections, curl http://localhost:6333/collections

# 3. 임베딩 생성 후 Qdrant Collection에 Insert/Update
# - 임베딩 모델 로드: 온라인, 오프라인 사용
# - 컬렉션에 데이터 삽입: Batch 단위로 Qdrant의 update/insert

# 4. Qdrant 의미 기반 검색
# - 의미 기반 검색으로 관련 문서 조회

# 5. QA 처리
# - Qdrant 검색 결과 -> QA 토크나이저 + QA 모델
# - 형태소 분석으로 한국어 처리 강화
# - 문자열 -> 토큰 ID 변환, 예시: "AI는 의료 분야에서 활용된다." -> [101, 1234, 5678, ...]
# - 모델 입력: input_ids 토큰 ID 배열, attention_mask 패딩 여부 표시, 모델 출력을 텍스트로 복원 [101, 1234, 5678, ...] -> "AI는 의료 분야에서 활용된다."

# 6. 요약 처리
# - 요약 토크나이저 + 요약 모델 (KoBART 기반)
# - 반복 억제, 길이 조절, 다양성 확보 등 파라미터 튜닝
# - 후처리(clean_summary)로 중복 제거
# - from transformers import BartTokenizer # 영어 전용이라 맞지 않음

# 7. 최종 응답: 외부 서비스 연계 검토
# - Local LLM은 GPU 장비 한계로 로컬 실행은 패스 -> 대신 Copilot에 문의하여 자연스러운 답변 재구성
# - 검토: 현재 로컬 GPU 장비로는 생성형 LLM 모델을 도메인에 맞추어 파인튜닝은 불가능, 현재 추론 모델도 좋은 성능의 모델로 교체 불가능

# 8. 서비스: 구글, 네이버 RAG 시스템 구성
# - /llm_app/transformer_rag2_23_app.py
# - FastAPI 구동 정보: 터미널에서 구동, uvicorn transformer_rag2_23_app:app --reload, 경로 포함 uvicorn LLM.llm_app.transformer_rag2_23_app:app --reload
# - FastAPI 서비스: /search, 입력: 질의문, 출력: QA + 요약 결과 + 출처 정보
# - 1)  [사용자 질의]
# - 2)  [FastAPI 엔드포인트: /search]
# - 3)  [SentenceTransformer: 임베딩]
# - 4)  [Qdrant 의미 기반 검색]
# - 5)  [검색 결과 문서]
# - 6)  [KoELECTRA QA 모델 + MeCab 후처리(형태소 분석: 한국어 처리 강황)]
# - 7)  [KoBART Summarization 모델 + clean_summary]
# - 8)  [응답 + 출처 표시]
# - 9)  [최종 응답 LLM 모델은  외부 서비스 연계 검토: 자연스러운 문장]
# - 10) [최종 사용자 응답]

In [4]:
# 데이터 수집 모듈 구현
import feedparser               # RSS 피드를 파싱해서 뉴스 항목을 가져옴
from bs4 import BeautifulSoup   # HTML 태그 제거 및 텍스트 정제
from datetime import datetime
import psycopg2                 # PostgreSQL DB 연결 및 SQL 실행
import logging                  # 실행 과정 기록 (파일 + 콘솔 출력)

# RSS 가져오기 함수
def fetch_rss(rss_url: str):
    # feedparser 파싱 -> 뉴스 항목(entries) 리스트 반환
    feed = feedparser.parse(rss_url)
    return feed.entries

# 뉴스 파싱 함수
def parse_entry(entry):
    title = entry.title # 뉴스 제목, RSS 항목의 <title> 태그에 해당
    content_raw = entry.description if "description" in entry else ""  # 뉴스 본문, RSS 항목에 description 속성, 없으면 빈 문자열 반환
    content = BeautifulSoup(content_raw, "html.parser").get_text() # 텍스트만 추출, HTML 제거
    url = entry.link # 뉴스 원문 링크(URL), RSS 항목의 <link> 태그
    published_at = datetime(*entry.published_parsed[:6]) if hasattr(entry, "published_parsed") else datetime.now() # entry.published_parsed가 있으면 datetime 객체로 변환(연, 월, 일, 시, 분, 초)
    source_name = entry.source.title if hasattr(entry, "source") else "Google News" # entry.source가 있으면 그 안의 title을 사용하고, 없으면 기본값 "Google News"로 설정
    source_url = entry.source.href if hasattr(entry, "source") else "" # entry.source가 있으면 href 속성을 사용하고, 없으면 빈 문자열로 처리

    return title, content, url, published_at, source_name, source_url

# DB Insert 함수
def insert_article(cur, conn, title, content, url, published_at, source_name, source_url):
    try: # SQL 실행
        cur.execute("""
            INSERT INTO news_articles (title, content, url, published_at, source_name, source_url)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (url) DO NOTHING;
        """, (title, content, url, published_at, source_name, source_url))
        if cur.rowcount > 0: # cur.rowcount → SQL 실행 후 영향을 받은 행 수
            logging.info("데이터 Insert 성공: %s", title) # 1 이상이면 새로운 데이터가 추가된 것 → “Insert 성공” 로그 기록
        else:
            logging.info("중복으로 건너뜀: %s", title) # 0이면 중복으로 인해 추가되지 않은 것 → “중복으로 건너뜀” 로그 기록
        conn.commit() # 실행 결과를 실제 DB 반영
    except Exception as e:
        logging.error("Insert 실패: %s, 에러: %s", title, e)

# 로깅 설정: 파일 + 콘솔 출력
logging.basicConfig(
    level=logging.INFO, # INFO 이상 레벨 기록
    format="%(asctime)s [%(levelname)s] %(message)s", # 출력 형식
    handlers=[
        logging.FileHandler("24.transformer_rag3_app.log", encoding="utf-8"), # 파일 저장
        logging.StreamHandler() # 콘솔 출력
    ]
)
    
# RSS 함수 호출
rss_url = "https://news.google.com/rss?hl=ko&gl=KR&ceid=KR:ko"
entries = fetch_rss(rss_url)

# DB 연결
conn = psycopg2.connect(dbname="newsdb", user="newsuser", password="1234", host="localhost", port="5432")
cur = conn.cursor()

# 뉴스 파싱 함수 -> DB Insert 함수
for entry in entries:
    title, content, url, published_at, source_name, source_url = parse_entry(entry)
    insert_article(cur, conn, title, content, url, published_at, source_name, source_url)

cur.close()
conn.close()
logging.info("DB 연결 종료 완료")

2026-05-08 12:58:22,410 [INFO] 중복으로 건너뜀: 미-이란, 협상 진전 하루 만에 교전…트럼프 “휴전 여전히 유효” - 한겨레
2026-05-08 12:58:22,414 [INFO] 중복으로 건너뜀: [속보] ‘채상병 순직’ 임성근 前 사단장, 1심 징역 3년 - 조선일보
2026-05-08 12:58:22,418 [INFO] 중복으로 건너뜀: 미 법원, 글로벌 10% 관세도 무효 판결…영향은 제한적일 듯 - 경향신문
2026-05-08 12:58:22,423 [INFO] 중복으로 건너뜀: 트럼프, 한국 시각 9일 백악관 연설…휴전 한 달, 이란 언급할까 - KBS 뉴스
2026-05-08 12:58:22,431 [INFO] 중복으로 건너뜀: 나무호, 두바이 도착…화재 원인 조사 본격화 - KBS 뉴스
2026-05-08 12:58:22,436 [INFO] 중복으로 건너뜀: 국힘 광역후보들 ‘조작기소 특검법’ 맹비난…정진석 “불출마” - KBS 뉴스
2026-05-08 12:58:22,442 [INFO] 중복으로 건너뜀: 한덕수 ‘내란 혐의’ 항소심 오늘 선고…징역 23년 유지될까 - 한겨레
2026-05-08 12:58:22,458 [INFO] 중복으로 건너뜀: "제가 대구시장 되면"…김부겸·추경호, 사흘째 SNS '신경전' - 노컷뉴스
2026-05-08 12:58:22,466 [INFO] 중복으로 건너뜀: 김정은, '서울 사정권' 신형 곡사포 점검‥"연내 남부 국경 배치" - MBC 뉴스
2026-05-08 12:58:22,481 [INFO] 중복으로 건너뜀: 개헌안 표결, 국힘 전원 불참에 무산…우원식 “내일 재표결” - 한겨레
2026-05-08 12:58:22,485 [INFO] 중복으로 건너뜀: “美 해상봉쇄 뚫었다”…이란, ‘유조선 환적’으로 3조원 챙겼다 - v.daum.net
2026-05-08 12:58:22,512 [INFO] 중복으로 건너뜀: 이번엔 성모마리아상 입에 담뱃불 ‘경악’···이스라엘군, 레바논서 또 신

In [ ]:
# 뉴스 조회
import psycopg2
import logging

# DB 연결
conn = psycopg2.connect(dbname="newsdb", user="newsuser", password="1234", host="localhost", port="5432")
cur = conn.cursor()

# 1. 전체 뉴스 개수 조회
cur.execute("""
            SELECT COUNT(*) 
            FROM news_articles;
""")
count = cur.fetchone()[0]
logging.info("총 뉴스 개수: %s", count)

# 2. 최신 뉴스 5개 조회
cur.execute("""
            SELECT title, published_at
            FROM news_articles
            ORDER BY published_at DESC
            LIMIT 5;
""")
lastest_news = cur.fetchall()
logging.info("최신 뉴스 5개:")
for row in lastest_news:
    logging.info("제목: %s | 발행일: %s", row[0], row[1])

# 3. 특정 키워드 검색(예시: AI)
keyword = 'AI'
cur.execute("""
            SELECT title, url
            FROM news_articles
            WHERE content LIKE %s
            ORDER BY published_at DESC
            LIMIT 10;
""", (f"%{keyword}%",)) # 튜플 형식 지켜야 에러가 안남
keyword_news = cur.fetchall()
logging.info("키워드 '%s' 관련 뉴스:", keyword)
for row in keyword_news:
    logging.info("제목: %s | URL: %s", row[0], row[1])

# 4. 출처별 뉴스 개수
cur.execute(""" 
            SELECT source_name, COUNT(*)
            FROM news_articles
            GROUP BY source_name
            ORDER BY COUNT(*) DESC;
""")
source_stats = cur.fetchall()
logging.info("출처별 뉴스 개수:")
for row in source_stats:
    logging.info("출처: %s | 개수: %s", row[0], row[1])

2026-05-08 13:52:13,605 [INFO] 총 뉴스 개수: 167
2026-05-08 13:52:13,610 [INFO] 최신 뉴스 5개:
2026-05-08 13:52:13,611 [INFO] 제목: 나무호, 두바이 도착…화재 원인 조사 본격화 - KBS 뉴스 | 발행일: 2026-05-08 03:12:00
2026-05-08 13:52:13,613 [INFO] 제목: 트럼프, 한국 시각 9일 백악관 연설…휴전 한 달, 이란 언급할까 - KBS 뉴스 | 발행일: 2026-05-08 02:50:00
2026-05-08 13:52:13,617 [INFO] 제목: [가요소식] 코르티스 미니 2집, 발매 4일만 '더블 밀리언셀러' - 연합뉴스 | 발행일: 2026-05-08 02:47:24
2026-05-08 13:52:13,619 [INFO] 제목: ‘한동훈 후원회장’ 정형근 “비상계엄, 윤석열이 잘하려다…한동훈은 ‘민주당 분대장’처럼 행동” - 한겨레 | 발행일: 2026-05-08 02:43:00
2026-05-08 13:52:13,622 [INFO] 제목: ‘포켓몬 생태도감’ 가장 큰 독자는 20대 여성 - 경향신문 | 발행일: 2026-05-08 02:24:00
2026-05-08 13:52:13,631 [INFO] 키워드 'AI' 관련 뉴스:
2026-05-08 13:52:13,637 [INFO] 제목: 국제유가, 미·이란 협상진전 신중론 속 하락…브렌트 100.1달러 - 한국무역협회-KITA.NET | URL: https://news.google.com/rss/articles/CBMi1gFBVV95cUxNUU05ZS1iZmRfTllNaWc4dmttN25CMl94RTZ6MEpTTkF1Nzdia2Fwc1lvYzBHT1J6TDg4bkFOVlFGT0JwYXBzMnlzMHhfMGNjY0JiMzVIQll4UEFsQVNPQ3luR291TFRWUzVaRG9rVjBmanFpMmNkNzFnR2RqWUZtaHBvTkpzbGJhWXFfc29JTkE2aWpvMk5

SyntaxError: syntax error at or near "("
LINE 2:             SELECT source_name COUNT(*)
                                            ^


In [ ]:
# # 데이터 수집 및 저장
# # - 수집 데이터 -> PostgreSQL 테이블 생성 및 입력
# # - DB 조회 + 로깅 설정으로 데이터 관리
# import psycopg2
# import json
# import logging

# # 로깅 설정: 파일 저장
# logging.basicConfig(
#     level=logging.INFO, # 로그 레벨: INFO 이상 기록
#     format="%(asctime)s [%(levelname)s] %(message)s", # 로그 출력 형식
#     handlers=[
#         logging.FileHandler("24.transformer_rag3_app.log", encoding="utf-8"), # 로그를 파일에 저장
#         logging.StreamHandler() # 콘솔에도 출력
#     ]
# )
# # DB에서 수집데이터 조회
# def fetch_data():
#     try:
#         with psycopg2.connect(
#             dbname="newsdb",      # 생성한 DB 이름
#             user="newsuser",      # 생성한 사용자
#             password="1234",      # 설정한 비밀번호
#             host="localhost",     # 로컬에서 실행 중
#             port="5432"           # 기본 포트
#         ) as conn:
#             with conn.cursor() as cur:
#                 # 모든 컬럼 조회
#                 cur.execute("SELECT id, title, content, url, published_at, source_name, source_url FROM news_articles;") # SQL 쿼리 실행
#                 rows = cur.fetchall() # fetchall() 전체 결과를 가져옴

#                 # 컬럼 이름 가져오기
#                 col_names = [ desc[0] for desc in cur.description ]

#                 # 결과를 JSON 형태로 변환
#                 result = []
#                 for row in rows:
#                     row_dict = dict(zip(col_names, row)) # 결과를 dict 형태로 변환: 컬럼 이름과 데이터 매핑 처리
#                     row_dict["published_at"] = str(row_dict["published_at"]) # date 컬럽 -> 문자열 변환
#                     result.append(row_dict)
#                 logging.info("PostgreSQL 연결 및 조회 성공!")
#                 return result
#     except Exception as e:
#         logging.error("데이터 조회 중 에러 발생: %s", e)
#         return [] # 에러가 발생해도 함수가 항상 빈 리스트를 반환, 프로그램이 멈추지 않고 계속 진행 가능

# # DB에서 수집데이터 조회 실행
# db_data = fetch_data()
# # JSON 변환, ensure_ascii=False 한글 깨짐 방지, indent=4 JSON을 4칸 들여쓰기
# print(json.dumps(db_data, ensure_ascii=False, indent=4))

2026-05-06 13:20:51,414 [INFO] PostgreSQL 연결 및 조회 성공!


[
    {
        "id": 1,
        "title": "“이 대통령 ‘과도한 요구’ 발언은 LG U+ 얘기”···삼전 노조위원장의 ‘책임 돌리기’ - 경향신문",
        "content": "“이 대통령 ‘과도한 요구’ 발언은 LG U+ 얘기”···삼전 노조위원장의 ‘책임 돌리기’  경향신문李대통령 \"과하다\" 경고에도… 삼전 노조 \"우리 얘기 아냐\" 딴청  조선일보이 대통령 \"일터 안전만큼은 타협 없어…노동과 기업 상생의 길 열 것\"  대한민국 정책브리핑[연합뉴스 이 시각 헤드라인] - 15:00  연합뉴스李 “‘친노동은 반기업’ 이분법 깨야…일터 안전 결코 타협하지 않을 것”  v.daum.net",
        "url": "https://news.google.com/rss/articles/CBMiWkFVX3lxTFBUN1l3NG5tRzQ2S0VUMFVXQUtVd3o5TXZNNlJyTVJSUXh6Qk0tOTJ1ZnhEblBhQ0g2dC1hQmp6ZG15TnU3dzMwcm9mVUI3RXZwLW5OcFZFN3VjZ9IBX0FVX3lxTE42OE5EU2FVU1Q2dG5Jc3QzTm9JVEozM3lsZFEtSFRaVnZnRGo4a2JxWVZYZXJmZWFDd0p3RnFoMWkybUdnSXpGWjkxLUZmUWZfcVpJV2hUbDNTT0tCZ1N3?oc=5",
        "published_at": "2026-05-01 05:23:00",
        "source_name": "경향신문",
        "source_url": "https://www.khan.co.kr"
    },
    {
        "id": 2,
        "title": "퇴근 2시간 전 “해고”…노동절에도 100일째 길 위에 선 노동자들 - 한겨레",
        "content": "퇴근 2시간 전 “해고”…노동절에도 100일째 길 위에 선 노동자들  한겨레되찾은 노동절, 양대 노총 2만3000명 도심 집결…\"노동기본권 보장

In [10]:
# # Qdrant 의미 기반 검색 구축
# # - Qdrant 구축 + Google News RSS 뉴스 컬렉션 생성
# # - Qdrant 검색엔진 서버 실행(qdrant.exe): .\LLM\qdrant\qdrant.exe
# # - https://github.com/qdrant/qdrant/releases 공식사이트: qdrant-x86_64-pc-windows-msvc.zip, 압축해제 qdrant.exe 실행
# # - API(curl) 테스트: http://localhost:6333/collections, curl http://localhost:6333/collections

# import torch
# from sentence_transformers import SentenceTransformer
# from qdrant_client import QdrantClient
# from qdrant_client.http import models # Qdrant 컬렉션 및 검색 관련 설정 정의하는 데이터 모델(구조체 클래스) 로드
# # Qdrant 클라이언트 연결: Qdrant 검색엔진 서버 실행 및 서버 연결 확인
# qdrant = QdrantClient(host="localhost", port=6333)

# # Qdrant 컬렉션 생성
# # - 벡터 크기는 모델에 따라 다르다(sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 -> 384차원)
# # - 거리(metric)는 보통 COSINE 사용
# # 컬렉션 존재 여부 확인 후 생성
# if not qdrant.collection_exists("news_articles_collection"):
#     qdrant.create_collection(
#         collection_name="news_articles_collection",
#         vectors_config=models.VectorParams(
#             size=384, # 임베딩 벡터 차원: paraphrase-multilingual-MiniLM-L12-v2 임베딩 모델과 차원을 동일하게 맞추어야 한다
#             distance=models.Distance.COSINE # 유사도 계산 방식
#         )
#     )
#     print("Qdrant 컬렉션 생성 완료")
# else:
#     print("news_articles_collection 컬렉션이 존재합니다.")

In [9]:
# # 임베딩 생성 후 Qdrant Collection에 Insert/Update
# # - 임베딩 모델 로드: 온라인, 오프라인 사용
# # - 컬렉션에 데이터 삽입: Batch 단위로 Qdrant의 update/insert

# # 디바이스 설정
# device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# print(f"PyTorch Version: {torch.__version__}, Device: {device}")

# # 임베딩 모델 로드
# # - 여기서는 paraphrase-multilingual-MiniLM-L12-v2 모델을 사용했는데, 다국어(한국어 포함) 문장 의미를 잘 반영하는 임베딩을 생성한다
# # - 문장을 입력하면 의미 공간에서 가까운 벡터로 변환
# embedder = SentenceTransformer( # 온라인 상태(단, 로컬캐시에 저장)
#     "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
#     device=device
# )

# # 오프라인 사용 준비: 온라인 환경에서 모델 다운로드
# # - git lfs install
# # - git clone https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
# # embedder = SentenceTransformer( # 오프라인 상태(경로 지정에 저장)
# #     "./offline_models/paraphrase-multilingual-MiniLM-L12-v2",
# #     device=device
# # )


# # 컬렉션에 데이터 삽입
# def insert_to_qdrant(data):
#     ids = []
#     vectors = []
#     payloads = []
#     for item in data:
#         text = item["title"] + " " + item["content"] # 제목+내용 임베딩 대상 데이터
#         embedding = embedder.encode(text).tolist() # 제목+내용 임베딩

#         ids.append(item["id"]) # ID 값
#         vectors.append(embedding) # 벡터 데이터
#         payloads.append(item) # 원본 데이터
    
#     # qdrant.upsert()는 삽입(insert)+업데이트(update) 기능을 동시에 수행, 같은 ID가 있으면 덮어쓰고, 없으면 새로 추가한다
#     # { - 데이터 저장 형식
#     #     "id": 1,
#     #     "vector": [0.123, -0.456, 0.789, ...],
#     #     "payload": {
#     #         "title": "AI 의료 활용",
#     #         "content": "AI가 의료 분야에서 활용되는 사례...",
#     #         "date": "2026-03-11",
#     #         "author": "홍길동"
#     #     }
#     # }
#     qdrant.upsert(
#         collection_name="news_articles_collection",
#         points=models.Batch(
#             ids=ids,
#             vectors=vectors,
#             payloads=payloads # payloads(JSON) 형태로 저장
#         )
#     )
#     print("데이터 임베딩 및 Qdrant 저장 완료")

# # Batch 단위로 Qdrant의 update/insert 테스트
# batch_size = 50
# for i in range(0, len(db_data), batch_size):
#     chunk = db_data[i:i+batch_size] # 2개씩 잘라서 가져옴
#     insert_to_qdrant(chunk) # 잘라낸 청크를 Qdrant에 전달

In [8]:
# # Qdrant collection 정보 확인: 전체 조회 + 특정 ID 조회
# from qdrant_client import QdrantClient

# qdrant = QdrantClient(host="localhost", port=6333)

# # 컬렉션 정보: 컬렉션 생성시 설정된 정보 및 상태를 보여주는 메타데이터
# info = qdrant.get_collection("news_articles_collection")
# print(info)

# # 전체 데이터 조회: limit 지정
# points = qdrant.scroll(
#     collection_name="news_articles_collection",
#     limit=10 # 최대 10개 반환
# )
# print(f"전체 데이터 조회: limit 지정\n{points}")

# # 특정 ID 데이터 조회
# point = qdrant.retrieve(
#     collection_name="news_articles_collection",
#     ids=[350,360],
#     with_payload=True, # payload 정보 포함
#     with_vectors=False # vectors 정보 제외
# )
# print(f"특정 ID 데이터 조회:\n{point}")